# GPU Memory Benchmark for Generative Models

Benchmark GPU memory usage for the model weights of
generative models across multiple large language models
and precision settings, including fp16 models and
GPTQ-int4 variants of Llama3.2 and Qwen2.5.

Each model is loaded with HF Transformers, measured for
GPU memory usage, and then unloaded before the next
configuration is tested. This keeps the comparison
focused on how model family, precision, quantization,
and backend choice affect GPU memory usage.

For non-quantized models, the benchmark explicitly loads
models with dtype="float16". Without this setting,
from_pretrained may default to fp32, which can report
roughly twice the expected memory usage.

Use Hugging Face Transformers as the primary backend for
measuring model-weight memory, because it allocates GPU
memory as needed when the model is loaded.

vLLM is less suitable for measuring only the memory used
by model weights. A vLLM instance reserves a GPU memory
budget controlled by gpu_memory_utilization, and that
budget covers both the model weights and the KV cache,
along with other runtime overhead. As a result, the
reported GPU memory usage reflects the full vLLM memory
budget rather than just the model weights.

Optionally include vLLM measurements to verify that the
observed GPU memory usage is consistent with the
configured gpu_memory_utilization budget.

In [ ]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)
logging.disable(logging.CRITICAL)

import warnings
warnings.filterwarnings("ignore")

import gc
import sys
sys.path.append("..")

import torch
from transformers import AutoModelForCausalLM
from gptqmodel import GPTQModel
from vllm import LLM

from unittests.notebook_utils import gpu_mem_used_gb, short_model_name

In [2]:
base_dir = '/groups/chichengz/tnn/datasets/'

model_names = [
    "Llama3.2-1B-Instruct",
    "Llama3.2-3B-Instruct",
    "Llama3.2-3B-Instruct-GPTQ",
    "Qwen2.5-3B-Instruct",
    "Qwen2.5-3B-Instruct-GPTQ-Int4",
    "Qwen2.5-7B-Instruct",
    "Qwen2.5-7B-Instruct-GPTQ-Int4",
    "Qwen2.5-Math-1.5B-Instruct",
    "Qwen2.5-Math-7B-Instruct",
    # "Llama3.3-70B-Instruct-GPTQ",
]

### Measure with HuggingFace Transformers

In [ ]:
tf_results = []

for name in model_names:
    print(f"\n=== {name} ===")
    llm_dir = base_dir + name

    if "GPTQ" in name:
        llm_tf = GPTQModel.load(llm_dir, device="cuda:0")
    else:
        llm_tf = AutoModelForCausalLM.from_pretrained(
            llm_dir,
            dtype="float16",
            device_map="cuda:0",
            trust_remote_code=True,
        )
        llm_tf.eval()

    mem_gb = gpu_mem_used_gb()
    print(f'  memory: {mem_gb:.2f} GB')
    tf_results.append((short_model_name(name), mem_gb))

    del llm_tf
    gc.collect()
    torch.cuda.empty_cache()

print("\n=== Summary (transformers) ===")
print(f"{'model':<22} {'memory (GB)':>12}")
print("-" * 35)
for name, mem in tf_results:
    print(f"{name:<22} {mem:>12.2f}")

### Measure with vLLM (optional)
vLLM pre-allocates KV cache, so memory usage depends on `gpu_memory_utilization`.

In [4]:
# vllm_results = []

# for name in model_names:
#     print(f"\n=== {name} ===")
#     llm_dir = base_dir + name

#     llm_vllm = LLM(
#         model=llm_dir,
#         tensor_parallel_size=1,
#         max_model_len=5000,
#         gpu_memory_utilization=0.25,
#         enforce_eager=True,
#         distributed_executor_backend=None,
#         disable_log_stats=True,
#         dtype="float16",
#         seed=0,
#     )

#     mem_gb = get_gpu_memory_used()
#     print(f'  memory: {mem_gb:.2f} GB')
#     vllm_results.append((name, mem_gb))

#     del llm_vllm
#     gc.collect()
#     torch.cuda.empty_cache()

# print("\n=== Summary (vLLM) ===")
# print(f"{'model':<35} {'memory (GB)':>12}")
# print("-" * 48)
# for name, mem in vllm_results:
#     print(f"{name:<35} {mem:>12.2f}")